In [ ]:
!git clone -b assignment2 https://token@github.com/mehtavirti/gnr638.git
%cd gnr638
!git fetch origin assignment2:assignment2
!git checkout assignment2

Cloning into 'gnr638'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 161 (delta 55), reused 125 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 3.83 MiB | 13.44 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/gnr638/gnr638/gnr638/gnr638
fatal: Refusing to fetch into current branch refs/heads/assignment2 of non-bare repository
Already on 'assignment2'
Your branch is up to date with 'origin/assignment2'.


In [ ]:
import sys
sys.path.insert(0, '/content/gnr638')
print("Repo cloned and on assignment-2 branch!")

Repo cloned and on assignment-2 branch!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install timm thop fvcore umap-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import zipfile

zip_path = '/content/train_data.zip'
extract_path = '/content/drive/MyDrive/AID'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done!")

Done!


In [ ]:
%cd /content/gnr638
!git fetch --all
!git checkout assignment-2
!git pull

/content/gnr638
Fetching origin
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 436 bytes | 436.00 KiB/s, done.
From https://github.com/mehtavirti/gnr638
   6b94dfa..ef31681  assignment2 -> origin/assignment2
error: pathspec 'assignment-2' did not match any file(s) known to git
Updating 6b94dfa..ef31681
Fast-forward
 models/load_models.py | 19 ++++++++-----------
 1 file changed, 8 insertions(+), 11 deletions(-)


In [ ]:
# Cell 4 — Verify dataset
import os

DATA_DIR = '/content/drive/MyDrive/AID/train_data'
classes  = sorted(os.listdir(DATA_DIR))
print(f"Number of classes found: {len(classes)}")
print(f"Classes: {classes}")

Number of classes found: 30
Classes: ['Airport', 'BareLand', 'BaseballField', 'Beach', 'Bridge', 'Center', 'Church', 'Commercial', 'DenseResidential', 'Desert', 'Farmland', 'Forest', 'Industrial', 'Meadow', 'MediumResidential', 'Mountain', 'Park', 'Parking', 'Playground', 'Pond', 'Port', 'RailwayStation', 'Resort', 'River', 'School', 'SparseResidential', 'Square', 'Stadium', 'StorageTanks', 'Viaduct']


In [ ]:
RESULTS_DIR = '/content/drive/MyDrive/results/scenario1'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

Results will be saved to: /content/drive/MyDrive/results/scenario1


In [ ]:
from utils.dataset_loader import get_dataloaders, get_fixed_pca_subset
from models.load_models   import load_model, freeze_backbone, compute_efficiency_metrics
from utils.metrics        import compute_accuracy

print("All common files imported successfully!")

All common files imported successfully!


In [ ]:
DRIVE_BASE  = '/content/drive/MyDrive/GNR638_A2'
DATA_DIR    = '/content/drive/MyDrive/AID/train_data'
RESULTS_DIR = f'{DRIVE_BASE}/results/scenario1'
CKPT_DIR    = f'{DRIVE_BASE}/checkpoints/scenario1'
LOGS_DIR    = f'{DRIVE_BASE}/logs/scenario1'

for d in [RESULTS_DIR, CKPT_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Verify dataset
classes_found = sorted(os.listdir(DATA_DIR))
print(f"Dataset found: {len(classes_found)} classes")
print(f"Results  → {RESULTS_DIR}")
print(f"Checkpts → {CKPT_DIR}")
print(f"Logs     → {LOGS_DIR}")

Dataset found: 30 classes
Results  → /content/drive/MyDrive/GNR638_A2/results/scenario1
Checkpts → /content/drive/MyDrive/GNR638_A2/checkpoints/scenario1
Logs     → /content/drive/MyDrive/GNR638_A2/logs/scenario1


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics        import confusion_matrix, silhouette_score
from sklearn.metrics        import pairwise_distances
from sklearn.decomposition  import PCA
from sklearn.manifold       import TSNE

from utils.dataset_loader import get_dataloaders, get_fixed_pca_subset
from models.load_models   import (load_model, freeze_backbone,
                                   compute_efficiency_metrics, count_parameters)
from utils.metrics        import compute_accuracy

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED        = 42
NUM_CLASSES = 30
EPOCHS      = 30
LR          = 1e-3
BATCH_SIZE  = 64
MODEL_NAMES = ['resnet50', 'densenet121', 'efficientnet_b0']

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device : {DEVICE}")
print(f"Models : {MODEL_NAMES}")

Device : cuda
Models : ['resnet50', 'densenet121', 'efficientnet_b0']


In [ ]:

efficiency_records = []

for name in MODEL_NAMES:
    print(f"\n{'='*50}\n  Model: {name}\n{'='*50}")
    model = load_model(name, num_classes=NUM_CLASSES, pretrained=True)
    model = freeze_backbone(model)
    metrics = compute_efficiency_metrics(model, input_size=(1,3,224,224), device='cpu')
    total_p, trainable_p = count_parameters(model)
    pct = 100.0 * trainable_p / total_p
    print(f"  Total params     : {total_p/1e6:.2f} M")
    print(f"  Trainable params : {trainable_p/1e6:.4f} M  ({pct:.2f}%)")
    efficiency_records.append({
        'Model':                name,
        'Total Params (M)':     round(total_p/1e6,     2),
        'Trainable Params (M)': round(trainable_p/1e6, 4),
        'Trainable %':          round(pct,              2),
        'MACs (G)':             metrics['macs_G'],
        'FLOPs (G)':            metrics['flops_G'],
    })
    del model
df_eff = pd.DataFrame(efficiency_records)
df_eff.to_csv(f'{RESULTS_DIR}/model_efficiency.csv', index=False)
print(f"\n── Efficiency Table ──────────────────────────────────")
print(df_eff.to_string(index=False))
print(f"\n Saved → {RESULTS_DIR}/model_efficiency.csv")


  Model: resnet50


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


  Efficiency Metrics
  Parameters : 23.57 M
  MACs       : 4.13 G
  FLOPs      : 4.11 G

  Total params     : 23.57 M
  Trainable params : 0.0615 M  (0.26%)

  Model: densenet121


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]


  Efficiency Metrics
  Parameters : 6.90 M
  MACs       : 2.83 G
  FLOPs      : 2.86 G

  Total params     : 6.98 M
  Trainable params : 0.0307 M  (0.44%)

  Model: efficientnet_b0


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]


  Efficiency Metrics
  Parameters : 4.00 M
  MACs       : 0.38 G
  FLOPs      : 0.40 G

  Total params     : 4.05 M
  Trainable params : 0.4480 M  (11.07%)

── Efficiency Table ──────────────────────────────────
          Model  Total Params (M)  Trainable Params (M)  Trainable %  MACs (G)  FLOPs (G)
       resnet50             23.57                0.0615         0.26      4.13       4.11
    densenet121              6.98                0.0307         0.44      2.83       2.86
efficientnet_b0              4.05                0.4480        11.07      0.38       0.40

 Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/model_efficiency.csv


In [ ]:


def train_one_model(model_name):
    """
    Trains linear probe for one model.
    Saves checkpoint to Drive after every epoch.
    Saves training log CSV after every epoch.
    Returns summary dict (no model object — freed from RAM after saving).
    """
    print(f"\n{'='*55}")
    print(f"  LINEAR PROBE TRAINING — {model_name.upper()}")
    print(f"{'='*55}")

    # ── Check if already done (resume protection) ─────────────────────────────
    summary_path = f'{RESULTS_DIR}/{model_name}_summary.csv'
    if os.path.exists(summary_path):
        print(f"Already trained. Loading saved summary.")
        return pd.read_csv(summary_path).to_dict('records')[0]

    # ── Data ──────────────────────────────────────────────────────────────────
    train_loader, val_loader, classes = get_dataloaders(
        DATA_DIR, batch_size=BATCH_SIZE, seed=SEED, augment=True)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = load_model(model_name, num_classes=NUM_CLASSES, pretrained=True)
    model = freeze_backbone(model)
    model = model.to(DEVICE)

    # Print + save efficiency at training start
    print(f"\n[Efficiency — {model_name}]")
    metrics  = compute_efficiency_metrics(model, device=str(DEVICE).split(':')[0])
    total_p, trainable_p = count_parameters(model)
    eff_row = {
        'model': model_name,
        'total_params_M':     round(total_p/1e6,     2),
        'trainable_params_M': round(trainable_p/1e6, 4),
        'trainable_pct':      round(100*trainable_p/total_p, 2),
        'macs_G':  metrics['macs_G'],
        'flops_G': metrics['flops_G'],
    }
    pd.DataFrame([eff_row]).to_csv(
        f'{LOGS_DIR}/{model_name}_efficiency.csv', index=False)  # SAVE

    # ── Optimizer ─────────────────────────────────────────────────────────────
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # ── Check for partial checkpoint (resume if disconnected) ─────────────────
    log_path  = f'{LOGS_DIR}/{model_name}_training_log.csv'
    ckpt_path = f'{CKPT_DIR}/{model_name}_latest.pth'
    start_epoch = 1
    train_log   = []
    best_val_acc = 0.0

    if os.path.exists(ckpt_path):
        print(f"\n Resuming from checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['state_dict'])
        optimizer.load_state_dict(ckpt['optimizer'])
        start_epoch  = ckpt['epoch'] + 1
        best_val_acc = ckpt['best_val_acc']
        if os.path.exists(log_path):
            train_log = pd.read_csv(log_path).to_dict('records')
        print(f"Resumed from epoch {start_epoch-1} | best_val={best_val_acc:.4f}")

    # ── Training Loop ─────────────────────────────────────────────────────────
    scenario_start = time.time()

    for epoch in range(start_epoch, EPOCHS + 1):
        epoch_start = time.time()

        # TRAIN
        model.train()
        t_correct, t_total, t_loss = 0, 0, 0.0
        for imgs, labels in tqdm(train_loader,
                                  desc=f"  [{model_name}] Epoch {epoch}/{EPOCHS}",
                                  leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            _, preds = torch.max(out, 1)
            t_correct += (preds == labels).sum().item()
            t_total   += labels.size(0)
            t_loss    += loss.item()

        train_acc = t_correct / t_total
        avg_loss  = t_loss / len(train_loader)

        # VALIDATE
        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                _, preds = torch.max(out, 1)
                v_correct += (preds == labels).sum().item()
                v_total   += labels.size(0)

        val_acc    = v_correct / v_total
        epoch_time = time.time() - epoch_start

        print(f"  Epoch [{epoch:02d}/{EPOCHS}] | "
              f"Loss: {avg_loss:.4f} | "
              f"Train: {train_acc:.4f} | "
              f"Val: {val_acc:.4f} | "
              f"Time: {epoch_time:.1f}s")

        # ── SAVE LOG AFTER EVERY EPOCH ────────────────────────────────────────
        train_log.append({
            'epoch': epoch, 'train_acc': round(train_acc, 4),
            'val_acc': round(val_acc, 4), 'train_loss': round(avg_loss, 4),
            'epoch_time_s': round(epoch_time, 1),
        })
        pd.DataFrame(train_log).to_csv(log_path, index=False)  # SAVE EVERY EPOCH

        # ── SAVE LATEST CHECKPOINT AFTER EVERY EPOCH ─────────────────────────
        torch.save({
            'epoch':        epoch,
            'model_name':   model_name,
            'state_dict':   model.state_dict(),
            'optimizer':    optimizer.state_dict(),
            'val_acc':      val_acc,
            'best_val_acc': max(best_val_acc, val_acc),
        }, ckpt_path)  # SAVE EVERY EPOCH

        # ── SAVE BEST CHECKPOINT SEPARATELY ──────────────────────────────────
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch':      epoch,
                'model_name': model_name,
                'state_dict': model.state_dict(),
                'val_acc':    val_acc,
            }, f'{CKPT_DIR}/{model_name}_best.pth')  # SAVE BEST
            print(f"Best checkpoint saved (val={val_acc:.4f})")

    total_time = time.time() - scenario_start

    # ── SAVE FINAL SUMMARY FOR THIS MODEL ─────────────────────────────────────
    summary = {
        'model':              model_name,
        'best_val_acc':       round(best_val_acc, 4),
        'final_train_acc':    round(train_acc,    4),
        'final_val_acc':      round(val_acc,      4),
        'train_val_gap':      round(train_acc - val_acc, 4),
        'total_time_min':     round(total_time/60, 2),
        'epochs':             EPOCHS,
        'lr':                 LR,
        'batch_size':         BATCH_SIZE,
        'device':             str(DEVICE),
    }
    pd.DataFrame([summary]).to_csv(summary_path, index=False)  # SAVE
    print(f"\n  Training complete | Best Val: {best_val_acc:.4f} | "
          f"Time: {total_time/60:.1f} min")
    print(f" Summary saved → {summary_path}")

    # Free model from RAM
    del model, optimizer
    torch.cuda.empty_cache()

    return summary


# ── RUN ALL 3 MODELS SEQUENTIALLY ─────────────────────────────────────────────
all_summaries = {}
for name in MODEL_NAMES:
    s = train_one_model(name)
    all_summaries[name] = s

# Combined summary table — SAVE
df_all_summary = pd.DataFrame(list(all_summaries.values()))
df_all_summary.to_csv(f'{RESULTS_DIR}/all_models_summary.csv', index=False)
print("\n── All Models Summary ──────────────────────────────────")
print(df_all_summary[['model','best_val_acc','final_train_acc',
                       'train_val_gap','total_time_min']].to_string(index=False))
print(f"Saved → {RESULTS_DIR}/all_models_summary.csv")


  LINEAR PROBE TRAINING — RESNET50


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Dataset] Train: 5594 | Val: 1399 | Fraction used: 1.0



[Efficiency — resnet50]



  Efficiency Metrics
  Parameters : 23.57 M
  MACs       : 4.13 G
  FLOPs      : 4.11 G


 Resuming from checkpoint: /content/drive/MyDrive/GNR638_A2/checkpoints/scenario1/resnet50_latest.pth
Resumed from epoch 1 | best_val=0.6819


...

  Epoch [26/30] | Loss: 0.0250 | Train: 0.9957 | Val: 0.9064 | Time: 89.3s


  Epoch [27/30] | Loss: 0.0335 | Train: 0.9923 | Val: 0.8978 | Time: 87.8s


  Epoch [28/30] | Loss: 0.0291 | Train: 0.9937 | Val: 0.9042 | Time: 88.5s


  Epoch [29/30] | Loss: 0.0295 | Train: 0.9932 | Val: 0.8978 | Time: 88.7s


  Epoch [30/30] | Loss: 0.0332 | Train: 0.9916 | Val: 0.8956 | Time: 88.6s

  Training complete | Best Val: 0.9114 | Time: 45.1 min
 Summary saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/efficientnet_b0_summary.csv

── All Models Summary ──────────────────────────────────
          model  best_val_acc  final_train_acc  train_val_gap  total_time_min
       resnet50        0.8763           0.9289         0.0525           45.61
    densenet121        0.9085           0.9794         0.0809           47.71
efficientnet_b0        0.9114           0.9916         0.0960           45.13
Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/all_models_summary.csv


In [ ]:

colors = {'resnet50': 'steelblue', 'densenet121': 'tomato',
          'efficientnet_b0': 'seagreen'}

# Load all training logs from Drive
logs = {}
for name in MODEL_NAMES:
    log_path = f'{LOGS_DIR}/{name}_training_log.csv'
    if os.path.exists(log_path):
        logs[name] = pd.read_csv(log_path)
        print(f"Loaded log: {name} ({len(logs[name])} epochs)")
    else:
        print(f"Missing: {log_path}")

# ── Plot 1: Train + Val Accuracy ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, df in logs.items():
    axes[0].plot(df['epoch'], df['train_acc'],
                 label=name, color=colors[name], linewidth=2)
    axes[1].plot(df['epoch'], df['val_acc'],
                 label=name, color=colors[name], linewidth=2)

for ax, title in zip(axes, ['Training Accuracy', 'Validation Accuracy']):
    ax.set_title(title, fontsize=13); ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy'); ax.legend(); ax.grid(True, alpha=0.4)
    ax.set_ylim(0, 1)
plt.suptitle('Scenario 1 — Linear Probe Accuracy Curves', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/plot_accuracy_curves.png',     # SAVE
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved plot_accuracy_curves.png")

# ── Plot 2: Training Loss ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
for name, df in logs.items():
    ax.plot(df['epoch'], df['train_loss'],
            label=name, color=colors[name], linewidth=2)
ax.set_title('Scenario 1 — Training Loss', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/plot_loss_curves.png', dpi=150, bbox_inches='tight')  # SAVE
plt.show()
print(f"Saved plot_loss_curves.png")

Loaded log: resnet50 (30 epochs)
Loaded log: densenet121 (30 epochs)
Loaded log: efficientnet_b0 (30 epochs)
Saved plot_accuracy_curves.png
Saved plot_loss_curves.png


In [ ]:
def load_checkpoint(model_name, ckpt_path):
    """Load checkpoint and strip thop-injected keys (total_ops, total_params)."""
    model = load_model(model_name, num_classes=NUM_CLASSES, pretrained=False)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    clean = {k: v for k, v in ckpt['state_dict'].items()
             if not k.endswith(('total_ops', 'total_params'))}
    model.load_state_dict(clean, strict=False)
    return model

In [ ]:
def run_confusion_matrix(model_name):
    ckpt_path = f'{CKPT_DIR}/{model_name}_best.pth'
    out_png   = f'{RESULTS_DIR}/plot_confusion_{model_name}.png'
    out_csv   = f'{RESULTS_DIR}/per_class_accuracy_{model_name}.csv'

    model = load_checkpoint(model_name, f'{CKPT_DIR}/{model_name}_best.pth')
    model = model.to(DEVICE)
    model.eval()

    _, val_loader, classes = get_dataloaders(
        DATA_DIR, batch_size=BATCH_SIZE, seed=SEED, augment=False)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"  [{model_name}] Predictions"):
            out = model(imgs.to(DEVICE))
            _, preds = torch.max(out, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    per_cls_acc = cm.diagonal() / cm.sum(axis=1)

    fig, ax = plt.subplots(figsize=(18, 15))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                ax=ax, annot_kws={'size': 7})
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True',      fontsize=12)
    ax.set_title(f'Confusion Matrix — {model_name}  '
                 f'(Overall Acc: {np.mean(per_cls_acc):.4f})', fontsize=13)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0,  fontsize=8)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    df_cls = pd.DataFrame({'Class': classes,
                            'Accuracy': np.round(per_cls_acc, 4)})
    df_cls = df_cls.sort_values('Accuracy', ascending=True)
    df_cls.to_csv(out_csv, index=False)

    print(f"  Saved → {out_png}")
    print(f"  Saved → {out_csv}")
    print(f"\n  ── Top 5 Failure Classes [{model_name}] ──")
    print(df_cls.head(5).to_string(index=False))

    del model
    torch.cuda.empty_cache()


for name in MODEL_NAMES:
    run_confusion_matrix(name)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Dataset] Train: 5594 | Val: 1399 | Fraction used: 1.0


  [resnet50] Predictions:   0%|          | 0/22 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  [resnet50] Predictions: 100%|██████████| 22/22 [00:17<00:00,  1.25it/s]


  Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/plot_confusion_resnet50.png
  Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/per_class_accuracy_resnet50.csv

  ── Top 5 Failure Classes [resnet50] ──
  Class  Accuracy
 Center    0.6216
 School    0.6250
 Resort    0.6400
 Square    0.7234
Stadium    0.7647


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


...

  Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/plot_confusion_densenet121.png
  Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/per_class_accuracy_densenet121.csv

  ── Top 5 Failure Classes [densenet121] ──
  Class  Accuracy
 Resort    0.6600
 School    0.6875
 Square    0.8085
 Bridge    0.8163
Airport    0.8182


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Dataset] Train: 5594 | Val: 1399 | Fraction used: 1.0


  [efficientnet_b0] Predictions:   0%|          | 0/22 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  [efficientnet_b0] Predictions: 100%|██████████| 22/22 [00:17<00:00,  1.23it/s]


  Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/plot_confusion_efficientnet_b0.png
  Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/per_class_accuracy_efficientnet_b0.csv

  ── Top 5 Failure Classes [efficientnet_b0] ──
 Class  Accuracy
Resort    0.7000
Bridge    0.7143
Church    0.7500
Square    0.7660
School    0.7917


In [ ]:
def extract_and_save_features(model_name):
    feat_path = f'{LOGS_DIR}/{model_name}_features.npz'


    model = load_checkpoint(model_name, f'{CKPT_DIR}/{model_name}_best.pth')
    model = model.to(DEVICE)
    model.eval()

    pca_loader, classes = get_fixed_pca_subset(DATA_DIR, seed=SEED)

    features_list, labels_list = [], []

    def hook_fn(module, input, output):
        feat = output.detach().cpu()
        if feat.dim() == 4:
            feat = feat.mean(dim=[2, 3])
        features_list.append(feat)

    hook = None
    for name_m, module in model.named_modules():
        if isinstance(module, nn.AdaptiveAvgPool2d):
            hook = module.register_forward_hook(hook_fn)
            break
    if hook is None:
        modules = list(model.named_modules())
        hook = modules[-2][1].register_forward_hook(hook_fn)

    with torch.no_grad():
        for imgs, labels in tqdm(pca_loader,
                                  desc=f"  [{model_name}] Extracting features"):
            _ = model(imgs.to(DEVICE))
            labels_list.extend(labels.numpy())

    hook.remove()
    features = torch.cat(features_list, dim=0).numpy()
    labels   = np.array(labels_list)

    np.savez_compressed(feat_path, features=features, labels=labels)
    print(f"Features saved → {feat_path}  shape: {features.shape}")

    del model
    torch.cuda.empty_cache()
    return features, labels


def plot_embeddings(model_name, features, labels, classes):
    out_png = f'{RESULTS_DIR}/plot_embeddings_{model_name}.png'

    # removed skip check — always overwrites

    n_cls = len(np.unique(labels))
    cmap = matplotlib.colormaps.get_cmap('tab20').resampled(n_cls)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    pca      = PCA(n_components=2, random_state=SEED)
    pca_2d   = pca.fit_transform(features)
    var_exp  = pca.explained_variance_ratio_
    for c in range(n_cls):
        m = labels == c
        axes[0].scatter(pca_2d[m,0], pca_2d[m,1],
                        color=cmap(c), s=15, alpha=0.7, label=classes[c])
    axes[0].set_title(f'PCA — {model_name}\n'
                      f'PC1={var_exp[0]:.1%}  PC2={var_exp[1]:.1%}',
                      fontsize=11)
    axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
    axes[0].grid(True, alpha=0.3)

    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, max_iter=1000)
    tsne_2d = tsne.fit_transform(features)
    for c in range(n_cls):
        m = labels == c
        axes[1].scatter(tsne_2d[m,0], tsne_2d[m,1],
                        color=cmap(c), s=15, alpha=0.7, label=classes[c])
    axes[1].set_title(f't-SNE — {model_name}', fontsize=11)
    axes[1].set_xlabel('Dim 1'); axes[1].set_ylabel('Dim 2')
    axes[1].grid(True, alpha=0.3)

    handles = [plt.Line2D([0],[0], marker='o', color='w',
                           markerfacecolor=cmap(i), markersize=8,
                           label=classes[i]) for i in range(n_cls)]
    fig.legend(handles=handles, loc='center right',
               bbox_to_anchor=(1.13, 0.5), fontsize=7)

    plt.suptitle(f'Feature Embeddings — {model_name}', fontsize=13)
    plt.tight_layout()
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f"Saved → {out_png}")


_, classes = get_fixed_pca_subset(DATA_DIR, seed=SEED)

for name in MODEL_NAMES:
    feats, lbls = extract_and_save_features(name)
    plot_embeddings(name, feats, lbls, classes)

[PCA Subset] Total samples: 900 (30 classes x 30 samples)
[PCA Subset] Total samples: 900 (30 classes x 30 samples)


  [resnet50] Extracting features: 100%|██████████| 15/15 [00:13<00:00,  1.10it/s]


Features saved → /content/drive/MyDrive/GNR638_A2/logs/scenario1/resnet50_features.npz  shape: (900, 2048)
Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/plot_embeddings_resnet50.png
[PCA Subset] Total samples: 900 (30 classes x 30 samples)


  [densenet121] Extracting features: 100%|██████████| 15/15 [00:11<00:00,  1.25it/s]


Features saved → /content/drive/MyDrive/GNR638_A2/logs/scenario1/densenet121_features.npz  shape: (900, 1024)
Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/plot_embeddings_densenet121.png
[PCA Subset] Total samples: 900 (30 classes x 30 samples)


  [efficientnet_b0] Extracting features: 100%|██████████| 15/15 [00:09<00:00,  1.56it/s]


Features saved → /content/drive/MyDrive/GNR638_A2/logs/scenario1/efficientnet_b0_features.npz  shape: (900, 1280)
Saved → /content/drive/MyDrive/GNR638_A2/results/scenario1/plot_embeddings_efficientnet_b0.png


In [ ]:

sep_records = []

for name in MODEL_NAMES:
    feat_path = f'{LOGS_DIR}/{name}_features.npz'
    data      = np.load(feat_path)
    feats, lbls = data['features'], data['labels']

    # Silhouette score
    sil = silhouette_score(feats, lbls, metric='euclidean',
                           sample_size=min(500, len(lbls)),
                           random_state=SEED)

    # Inter / intra class distances
    unique_cls = np.unique(lbls)
    centroids  = np.array([feats[lbls==c].mean(axis=0) for c in unique_cls])
    dists      = pairwise_distances(centroids)
    np.fill_diagonal(dists, np.nan)
    mean_inter = np.nanmean(dists)

    intra = [pairwise_distances(feats[lbls==c]).mean()
             for c in unique_cls if len(feats[lbls==c]) > 1]
    mean_intra = np.mean(intra)

    sep_records.append({
        'Model':                    name,
        'Silhouette Score':         round(sil,              4),
        'Mean Inter-class Dist':    round(mean_inter,       4),
        'Mean Intra-class Dist':    round(mean_intra,       4),
        'Separability Ratio':       round(mean_inter/mean_intra, 4),
    })
    print(f"  {name}: sil={sil:.4f} | inter={mean_inter:.2f} | intra={mean_intra:.2f}")

df_sep = pd.DataFrame(sep_records)
df_sep.to_csv(f'{RESULTS_DIR}/feature_separability.csv', index=False)       # SAVE
print(f"\n── Separability Table ────────────────────────────────")
print(df_sep.to_string(index=False))
print(f"Saved → feature_separability.csv")

# ── Load all summaries from Drive and print final table ───────────────────────
summary_rows = []
for name in MODEL_NAMES:
    p = f'{RESULTS_DIR}/{name}_summary.csv'
    if os.path.exists(p):
        summary_rows.append(pd.read_csv(p).iloc[0].to_dict())

df_final = pd.DataFrame(summary_rows)
df_final.to_csv(f'{RESULTS_DIR}/scenario1_final_summary.csv', index=False)  # SAVE
print(f"\n── Final Summary Table ───────────────────────────────")
print(df_final[['model','best_val_acc','final_train_acc',
                'train_val_gap','total_time_min']].to_string(index=False))
print(f"Saved → scenario1_final_summary.csv")

# ── Computational Budget Report ───────────────────────────────────────────────
budget_rows = []
for name in MODEL_NAMES:
    eff_path = f'{LOGS_DIR}/{name}_efficiency.csv'
    sum_path = f'{RESULTS_DIR}/{name}_summary.csv'
    if os.path.exists(eff_path) and os.path.exists(sum_path):
        eff = pd.read_csv(eff_path).iloc[0]
        sm  = pd.read_csv(sum_path).iloc[0]
        budget_rows.append({
            'Model':           name,
            'Epochs':          int(sm['epochs']),
            'Time (min)':      sm['total_time_min'],
            'Device':          sm['device'],
            'Batch Size':      int(sm['batch_size']),
            'LR':              sm['lr'],
            'FLOPs (G)':       eff['flops_G'],
            'MACs (G)':        eff['macs_G'],
            'Trainable % ':    eff['trainable_pct'],
        })

df_budget = pd.DataFrame(budget_rows)
df_budget.to_csv(f'{RESULTS_DIR}/computational_budget.csv', index=False)    # SAVE
print(f"\n── Computational Budget ──────────────────────────────")
print(df_budget.to_string(index=False))
print(f"Saved computational_budget.csv")

  resnet50: sil=-0.0308 | inter=2.77 | intra=5.96
  densenet121: sil=0.0131 | inter=12.23 | intra=25.79
  efficientnet_b0: sil=0.0985 | inter=15.18 | intra=18.76

── Separability Table ────────────────────────────────
          Model  Silhouette Score  Mean Inter-class Dist  Mean Intra-class Dist  Separability Ratio
       resnet50           -0.0308                 2.7670               5.959300              0.4643
    densenet121            0.0131                12.2253              25.785999              0.4741
efficientnet_b0            0.0985                15.1785              18.761101              0.8090
Saved → feature_separability.csv

── Final Summary Table ───────────────────────────────
          model  best_val_acc  final_train_acc  train_val_gap  total_time_min
       resnet50        0.8763           0.9289         0.0525           45.61
    densenet121        0.9085           0.9794         0.0809           47.71
efficientnet_b0        0.9114           0.9916         0.09

In [ ]:

print("── All files on Google Drive ─────────────────────────\n")

for folder, label in [(RESULTS_DIR, 'RESULTS'),
                       (CKPT_DIR,    'CHECKPOINTS'),
                       (LOGS_DIR,    'LOGS')]:
    print(f"[{label}] {folder}")
    for f in sorted(os.listdir(folder)):
        size = os.path.getsize(f'{folder}/{f}')
        unit = 'KB' if size < 1e6 else 'MB'
        val  = size/1024 if size < 1e6 else size/(1024*1024)
        print(f"  {f:<55} {val:>7.1f} {unit}")
    print()

── All files on Google Drive ─────────────────────────

[RESULTS] /content/drive/MyDrive/GNR638_A2/results/scenario1
  all_models_summary.csv                                      0.3 KB
  computational_budget.csv                                    0.2 KB
  densenet121_summary.csv                                     0.2 KB
  efficientnet_b0_summary.csv                                 0.2 KB
  feature_separability.csv                                    0.2 KB
  model_efficiency.csv                                        0.2 KB
  per_class_accuracy_densenet121.csv                          0.5 KB
  per_class_accuracy_efficientnet_b0.csv                      0.5 KB
  per_class_accuracy_resnet50.csv                             0.5 KB
  plot_accuracy_curves.png                                   83.4 KB
  plot_confusion_densenet121.png                            224.3 KB
  plot_confusion_efficientnet_b0.png                        224.8 KB
  plot_confusion_resnet50.png                          